In [1]:
import slangpy as spy
from pyglm import glm
import matplotlib.pyplot as plt
import numpy as np

from bvhgs import device
from bvhgs.camera import Camera
from bvhgs.gaussian import GaussianCloud
from bvhgs.renderer import Renderer

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


In [2]:
np.random.seed(348)

# Create Gaussian Buffer

In [3]:
gaussians = GaussianCloud(16)
len(gaussians)

16

# Load Module and Shader

In [4]:
module = device.load_module("renderer.slang")
module

SlangModule(
  name = renderer.slang,
  path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
  entry_points = [
    SlangEntryPoint(name="project", stage=compute),
    SlangEntryPoint(name="cull", stage=compute),
    SlangEntryPoint(name="computeTile", stage=compute),
    SlangEntryPoint(name="buildGaussianTable", stage=compute),
    SlangEntryPoint(name="rasterize", stage=compute),
  ]
)

In [5]:
program = device.link_program([module], [])
program

ShaderProgram(
  modules = [
    SlangModule(
      name = renderer.slang,
      path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
      entry_points = [
        SlangEntryPoint(name="project", stage=compute),
        SlangEntryPoint(name="cull", stage=compute),
        SlangEntryPoint(name="computeTile", stage=compute),
        SlangEntryPoint(name="buildGaussianTable", stage=compute),
        SlangEntryPoint(name="rasterize", stage=compute),
      ]
    ),
  ],
  entry_points = []
)

# Projection

## Build Slang Buffer and Camera Parameter

In [6]:
# Create a buffer for the Gaussian points.
gaussian_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_3d,
    usage=spy.BufferUsage.shader_resource,
)
# Store all the gaussian points in the buffer.
gaussian_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_3d.type_layout.element_type_layout,
    gaussian_buf,
)

for i in range(len(gaussians)):
    print(gaussians[i]["position"])
    gaussian_cursor[i].write(gaussians[i])
gaussian_cursor.apply()

[-0.74514765 -1.4626689   1.5440476 ]
[-0.3148266   1.3637321   0.32085106]
[1.29413    0.0143156  0.14287208]
[-0.41008282 -1.1208322  -0.55690503]
[ 1.6357784  -1.0389293  -0.29886138]
[-0.39862242 -0.33556923 -1.24802   ]
[-0.6731114  0.607563   0.8623601]
[-1.7406042   0.29265207 -0.66594994]
[ 0.6454373 -1.2618792  0.7806917]
[ 0.86589104 -0.7342854  -0.9310714 ]
[0.89830947 0.43982178 1.5116702 ]
[ 1.471268   -0.41588905 -0.5301987 ]
[-1.4924484   0.9790674  -0.27851436]
[-0.5074454  -0.36497736 -0.6466633 ]
[0.3664213 0.6940648 1.8170619]
[-0.41561976  1.029824    0.5436981 ]


In [7]:
# Create a buffer for the Gaussian2D points.
gaussian2d_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_2d,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
gaussian2d_cursor[0].read()

{'position': {0, 0, 0},
 'covariance': {{0, 0}, {0, 0}},
 'color': {0, 0, 0},
 'opacity': 0.0,
 'cachedInvCov': {{0, 0}, {0, 0}},
 'cachedDet': 0.0,
 'cachedNorm': 0.0}

In [8]:
camera = Camera(
    rotation=glm.quat(1, 0, 0, 0),
    translation=glm.vec3(0, 0, 1),
    sensor_size=glm.uvec2(512, 512),
    focal_length=64
)
camera.to_slang()

{'_rotation': [0.0, 0.0, 0.0, 1.0],
 '_translation': vec3( 0, 0, 1 ),
 '_sensorSize': uvec2( 512, 512 ),
 '_focalLength': 64}

## Dispatch Projection Kernel

In [9]:
# Create flag buffer for culling.
cull_flag_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_flag,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [10]:
ker_proj = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("project")])
)
ker_proj

ComputeKernel(0x600002584ec0)

In [11]:
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf
    }
)

In [12]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{-0.29289848, -0.57493776, 3.027676}
{-0.23835133, 1.0324647, 1.9244554}
{1.1323489, 0.01252599, 1.7265961}
{-0.92549646, -2.5295532, 1.2730931}
{2.3330314, -1.4817746, 2.0607622}
{-398622.44, -335569.22, -0.5770793}
{-0.36142924, 0.32623285, 2.0713756}
{-5.210609, 0.8760725, 1.7963678}
{0.36246437, -0.7086455, 2.2759154}
{12.562145, -10.652842, 1.1374065}
{0.35765424, 0.1751113, 2.703496}
{3.1316814, -0.88524455, 1.5994709}
{-2.0685768, 1.3570158, 1.9252316}
{-1.4361526, -1.032945, 0.7180224}
{0.13007216, 0.24637897, 2.924351}
{-0.26923645, 0.6671149, 1.9016522}


## Cull Gaussians

In [13]:
prefix_sum = cull_flag_buf.to_numpy().view(np.int32)
prefix_sum = np.cumsum(prefix_sum).astype(np.int32)
print(prefix_sum)

[0 0 0 0 0 0 0 0 0 0 1 1 1 1 2 2]


In [14]:
num_culled_gaussians = prefix_sum[-1]
print(num_culled_gaussians)
cull_prefix_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_cull_prefix,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
cull_prefix_buf.copy_from_numpy(prefix_sum)

2


In [15]:
# Culled gaussians
culled_gaussian_buf = device.create_buffer(
    element_count=num_culled_gaussians,
    struct_type=program.reflection.g_gaussian_2d_culled,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [16]:
ker_cull = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("cull")])
)
ker_cull.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_gaussian_2d": gaussian2d_buf,
        "g_cull_flag": cull_flag_buf,
        "g_cull_prefix": cull_prefix_buf,
        "g_gaussian_2d_culled": culled_gaussian_buf
    }
)

In [17]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d_culled.type_layout.element_type_layout,
    culled_gaussian_buf,
)
for i in range(gaussian2d_cursor.element_count):
    print(gaussian2d_cursor[i].read()["position"])

{0.35765424, 0.1751113, 2.703496}
{0.13007216, 0.24637897, 2.924351}
